# Learning a swing up policy using reinforcement learning (Vanilla Q-learning)
by TimJanssen66

### General understanding
I'm using ...

## Table of contents

1. <a href="#Training-on-simulator">Training on simulator</a>
2. <a href="#Testing-on-simulator">Testing on simulator</a>
3. <a href="#Testing-on-setup">Testing on setup</a>
4. <a href="#Exercise-6:-Design-Assignment-Environment.">Exercise 6: Design Assignment Environment</a>


In [1]:
import numpy as np
import gymnasium as gym
import gym_unbalanced_disk
import pickle
from collections import defaultdict
import time
import matplotlib.pyplot as plt
import pickle

class CompatibilityWrapper(gym.Wrapper):
    """
    Translates old Gym API environments (like UnbalancedDisk) 
    to the modern Gymnasium API to prevent crash errors.
    """
    def reset(self, *, seed=None, options=None):
        # 1. Intercept the reset call and strip the unexpected kwargs
        obs = self.env.reset()
        
        # 2. Ensure the output is a tuple of (observation, info_dict)
        if not isinstance(obs, tuple):
            return obs, {}
        elif len(obs) == 2 and isinstance(obs[1], dict):
            return obs
        else:
            return obs, {}

    def step(self, action):
        result = self.env.step(action)
        
        # 1. If the old environment returns 4 variables (obs, reward, done, info)
        if len(result) == 4:
            obs, reward, done, info = result
            # Map 'done' to 'terminated', and set 'truncated' to False
            return obs, reward, done, False, info
            
        # 2. If it already returns 5, just pass it through
        return result

In [4]:
# --- 1. Custom Wrappers ---
class DiscretizeAction(gym.ActionWrapper):
    def __init__(self, env, discrete_actions):
        super().__init__(env)
        self.discrete_actions = np.array(discrete_actions)
        # E.g., 5 actions: 0, 1, 2, 3, 4 mapping to -3, -1.5, 0, 1.5, 3 Volts
        self.action_space = gym.spaces.Discrete(len(self.discrete_actions))

    def action(self, act_idx):
        # Extract the voltage and convert it to a pure Python float
        # This prevents shape mismatch errors in the SciPy differential equation solver
        return float(self.discrete_actions[act_idx])

class DiscretizeState(gym.ObservationWrapper):
    def __init__(self, env, nvec):
        super().__init__(env)
        self.nvec = np.array(nvec)
        self.observation_space = gym.spaces.MultiDiscrete(self.nvec)
        # Approximate physical bounds for [theta, omega]
        # Depending on the env, theta might wrap to [-pi, pi], and omega is bounded
        self.low = np.array([-np.pi, -40.0])  
        self.high = np.array([np.pi, 40.0])

    def observation(self, obs):
        # Ensure theta wraps between -pi and pi
        obs[0] = ((obs[0] + np.pi) % (2 * np.pi)) - np.pi
        
        # Clip velocity just in case it spins too fast
        obs = np.clip(obs, self.low, self.high)
        
        # Normalize and bin
        norm = (obs - self.low) / (self.high - self.low)
        disc = np.floor(norm * self.nvec).astype(int)
        disc = np.clip(disc, 0, self.nvec - 1)
        return tuple(disc)

## Training on simulator

In [5]:
# --- 2. Q-Learning Training Algorithm ---
def argmax_random(a):
    """Argmax that randomly breaks ties"""
    a = np.array(a)
    return np.random.choice(np.arange(len(a))[a == np.max(a)])

def train_q_learning(env, episodes=2000, alpha=0.2, gamma=0.99, eps_start=1.0, eps_end=0.01):
    Qmat = defaultdict(float)
    
    for ep in range(episodes):
        obs, _ = env.reset()
        terminated = truncated = False
        
        # Epsilon decay
        eps = max(eps_end, eps_start - (ep / (episodes * 0.8)))
        
        while not (terminated or truncated):
            # Epsilon-Greedy action selection
            if np.random.rand() < eps:
                action = env.action_space.sample()
            else:
                action = argmax_random([Qmat[obs, a] for a in range(env.action_space.n)])
                
            next_obs, reward, terminated, truncated, _ = env.step(action)
            
            # TD Update
            if terminated:
                TD = reward - Qmat[obs, action]
            else:
                Q_next_max = max([Qmat[next_obs, a] for a in range(env.action_space.n)])
                TD = reward + gamma * Q_next_max - Qmat[obs, action]
                
            Qmat[obs, action] += alpha * TD
            obs = next_obs
            
        if ep % 100 == 0:
            print(f"Episode {ep} completed. Epsilon: {eps:.2f}")
            
    return Qmat

# --- 3. Execution Script ---
if __name__ == "__main__":
    # 1. Create Base Environment DIRECTLY to avoid gym.make's hidden wrappers
    env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05)
    
    # 2. Translate the old 4-variable API to the new 5-variable API immediately
    env = CompatibilityWrapper(env)
    
    # 3. Apply standard TimeLimit manually
    env = gym.wrappers.TimeLimit(env, max_episode_steps=300)
    
    # 4. Apply Custom Wrappers: Restrict hardware strictly to [-3, 3] Volts
    allowed_voltages = [-3.0, -1.5, 0.0, 1.5, 3.0]
    env = DiscretizeAction(env, discrete_actions=allowed_voltages)
    
    # 5. Discretize State: 40 bins for Angle, 40 bins for Velocity
    env = DiscretizeState(env, nvec=[40, 40])
    
    print("Starting Training...")
    trained_Q = train_q_learning(env, episodes=1000)
    
    # Save the Q-table to a file so we can load it in the Test & Hardware scripts
    with open("q_table_disk.pkl", "wb") as f:
        pickle.dump(dict(trained_Q), f)
    print("Training finished. Q-table saved.")
    env.close()


Starting Training...
Episode 0 completed. Epsilon: 1.00
Episode 100 completed. Epsilon: 0.88
Episode 200 completed. Epsilon: 0.75
Episode 300 completed. Epsilon: 0.62
Episode 400 completed. Epsilon: 0.50
Episode 500 completed. Epsilon: 0.38
Episode 600 completed. Epsilon: 0.25
Episode 700 completed. Epsilon: 0.12
Episode 800 completed. Epsilon: 0.01
Episode 900 completed. Epsilon: 0.01
Training finished. Q-table saved.


## Test on simulator

In [6]:
# (Ensure DiscretizeAction and DiscretizeState classes are copied here too)

if __name__ == "__main__":
    # Setup environment exactly as in training
    env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05)
    env = CompatibilityWrapper(env)
    # (Note: TimeLimit isn't strictly needed for the manual test loop, but good for consistency)
    env = gym.wrappers.TimeLimit(env, max_episode_steps=300) 
    
    allowed_voltages = [-3.0, -1.5, 0.0, 1.5, 3.0]
    env = DiscretizeAction(env, discrete_actions=allowed_voltages)
    env = DiscretizeState(env, nvec=[40, 40])
    
    # Load the trained Q-table
    with open("q_table_disk.pkl", "rb") as f:
        Qmat = pickle.load(f)
        
    obs, _ = env.reset()
    try:
        env.render()
        for i in range(400):
            # Pure Exploitation (No Epsilon)
            # Find the best action for the current state in the loaded dictionary
            action = np.argmax([Qmat.get((obs, a), 0.0) for a in range(env.action_space.n)])
            
            obs, reward, terminated, truncated, _ = env.step(action)
            env.render()
            time.sleep(0.05) # Match the dt of the system
            
            if terminated or truncated:
                print("Episode finished.")
                break
    finally:
        env.close()

Episode finished.


## Test on setup

In [ ]:
# --- Setup Constants (Must perfectly match training) ---
nvec = np.array([40, 40])
allowed_voltages = [-3.0, -1.5, 0.0, 1.5, 3.0]
low = np.array([-np.pi, -40.0])  
high = np.array([np.pi, 40.0])

# Load Q-table
with open("q_table_disk.pkl", "rb") as f:
    Qmat = pickle.load(f)

# Define the physical math (same as the wrapper)
def discretize_physical_obs(theta, omega):
    obs = np.array([theta, omega])
    obs[0] = ((obs[0] + np.pi) % (2 * np.pi)) - np.pi # Wrap angle
    obs = np.clip(obs, low, high)
    
    norm = (obs - low) / (high - low)
    disc = np.floor(norm * nvec).astype(int)
    disc = np.clip(disc, 0, nvec - 1)
    return tuple(disc)

# --- Hardware Loop Template ---
# import your_university_hardware_api as hw

try:
    # hw.initialize()
    print("Executing on physical hardware...")
    
    for step in range(500): # Run for some time
        # 1. Read Sensors
        # theta, omega = hw.read_sensors()
        theta, omega = 0.0, 0.0 # Placeholder
        
        # 2. Discretize
        obs = discretize_physical_obs(theta, omega)
        
        # 3. Lookup best action index
        action_idx = np.argmax([Qmat.get((obs, a), 0.0) for a in range(len(allowed_voltages))])
        
        # 4. Map index back to real voltage
        voltage = allowed_voltages[action_idx]
        
        # 5. Send to motor
        # hw.write_voltage(voltage)
        
        # 6. Wait for next timestep (dt=0.05)
        time.sleep(0.05)

finally:
    # Always ensure motor is turned off when script ends or crashes!
    # hw.write_voltage(0.0)
    # hw.close()
    print("Safely shut down.")

# Exercise 6: Design Assignment Environment

In [ ]:
!pip install git+https://github.com/MaartenSchoukens/gym-unbalanced-disk@master
# in case of error try installing pyglet 
# !pip install pyglet


In [ ]:
import gymnasium as gym
import time
import gym_unbalanced_disk #is required
import numpy as np
env = gym.make('unbalanced-disk-v0',dt=0.05) #if this fails restart the kernel or use gym_unbalanced_disk.UnbalancedDisk
env = gym_unbalanced_disk.UnbalancedDisk(dt=0.05)
# I uncommented the last env = ..., Is that the right thing to do?


In [ ]:
#if you had error just run this cell again
observation, info = env.reset()
try:
    for i in range(200):
        u = env.action_space.sample() #random input
        observation, reward, terminated, truncated, info = env.step(u) 
        print(observation, reward)
        env.render()
        time.sleep(1/24)
        if terminated or truncated:
            observation, info = env.reset()
finally: #this will always run such to close the visualization
    env.close()

    

In [ ]:
Umax = 4

# c) Input sequence
# We create an open-loop input sequence that alternates maximum control effort.
# 20 steps * 0.05s = 1.0 seconds per swing direction.
ulist = np.concatenate([
    np.full(10, Umax),   # Swing right
    np.full(10, -Umax),  # Swing left
    np.full(10, Umax),   # Swing right (building momentum)
    np.full(10, -Umax),  # Swing left  (building momentum)
]) 
# This sequence actually swings back over the top and forth over the top
obs, info = env.reset()
try:
    for u in ulist:
        obs, reward, terminated, truncated, info = env.step(u)
        print(obs,reward)
        env.render()
        time.sleep(1/24)
        if terminated or truncated:
            obs, info = env.reset()
finally: #this will always run
    env.close()
    